# 附录 A：PyTorch 入门

> 全书的基础前置。主线 ch02-ch07 全用 PyTorch，本附录把核心概念过一遍：**张量 → autograd → nn.Module → 训练循环 → DataLoader → GPU**。

> 如果你已熟悉 PyTorch，可跳过本附录。这里是浓缩速查版。

## 1. 张量（Tensor）

张量是 PyTorch 的核心数据结构——多维数组，类似 NumPy 的 ndarray，但能在 GPU 上运算并支持自动求导。

In [ ]:
import torch

# 创建张量的几种方式
a = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)
b = torch.zeros(2, 3)         # 全零
c = torch.randn(2, 3)         # 标准正态随机
print(f"a:\n{a}")
print(f"shape={tuple(a.shape)}, dtype={a.dtype}, device={a.device}")

# 基本运算（逐元素、矩阵乘、广播）
d = torch.randn(2, 3)
sum_ = c + d                   # 逐元素加
matmul = a @ a.T               # 矩阵乘
print(f"\n逐元素加 shape: {tuple(sum_.shape)}, 矩阵乘 shape: {tuple(matmul.shape)}")
print(f"广播 (2,1)+(3,): {(torch.randn(2,1) + torch.randn(3)).shape}")

## 2. autograd 自动求导

PyTorch 的灵魂：**自动计算梯度**。设置 `requires_grad=True`，PyTorch 会记录所有运算，调用 `.backward()` 自动算出梯度。这是神经网络训练的基础。

In [ ]:
# autograd 示例：求 y = x² 的导数（应为 2x）
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()              # 反向传播，自动算梯度
print(f"y = x² 在 x=3 处的导数: {x.grad}（应为 2×3=6）")

# 神经网络场景：loss 对参数求梯度
w = torch.randn(3, requires_grad=True)    # 可训练参数
x = torch.randn(3)
loss = (w * x).sum() ** 2                 # 某个损失
loss.backward()
print(f"\nloss={loss.item():.4f}, w 的梯度: {w.grad}")

## 3. nn.Module 构建模型

`nn.Module` 是所有神经网络层的基类。继承它，在 `__init__` 里定义参数（层），在 `forward` 里定义前向计算。

In [ ]:
import torch.nn as nn

class SimpleNet(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)   # 自动注册参数
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, out_dim)
    
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

model = SimpleNet(10, 32, 2)
x = torch.randn(4, 10)         # batch=4, 特征=10
out = model(x)                 # 自动调用 forward
print(f"输出: {tuple(out.shape)}")
print(f"参数量: {sum(p.numel() for p in model.parameters()):,}")
# parameters() 自动收集所有 requires_grad=True 的参数
print(f"\n所有参数:")
for name, p in model.named_parameters():
    print(f"  {name}: {tuple(p.shape)}")

## 4. 训练循环（5 步范式）

几乎所有 PyTorch 训练都遵循这个范式：**前向 → 算 loss → 清梯度 → 反向 → 更新**。

In [ ]:
# 完整训练 demo：拟合 y = 2x + 1
torch.manual_seed(42)
model = nn.Linear(1, 1)                          # 待训练：y = wx + b
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)  # 优化器

# 造数据
X = torch.linspace(-1, 1, 100).unsqueeze(1)
Y = 2 * X + 1 + 0.1 * torch.randn(100, 1)        # y=2x+1 加噪声

for epoch in range(100):
    # ★ 5 步范式
    pred = model(X)                               # 1. 前向
    loss = nn.functional.mse_loss(pred, Y)        # 2. 算 loss
    optimizer.zero_grad()                         # 3. 清梯度
    loss.backward()                               # 4. 反向求梯度
    optimizer.step()                              # 5. 更新参数
    if epoch % 20 == 0:
        print(f"epoch {epoch}: loss {loss.item():.4f}")

print(f"\n学到: w={model.weight.item():.3f}（应为2）, b={model.bias.item():.3f}（应为1）")

## 5. DataLoader 批量加载数据

真实数据量很大，需分批训练。`Dataset` 定义单个样本，`DataLoader` 负责 batching/shuffling/并行加载。

In [ ]:
from torch.utils.data import Dataset, DataLoader

class ToyDataset(Dataset):
    def __init__(self, n=100):
        self.x = torch.randn(n, 4)
        self.y = torch.randint(0, 2, (n,))       # 二分类标签
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        return self.x[i], self.y[i]              # 返回单个样本

dataset = ToyDataset(100)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, drop_last=True)
print(f"数据集: {len(dataset)} 样本")
print(f"批次: {len(dataloader)} 个（每批 16）")

# 遍历 DataLoader
for i, (x_batch, y_batch) in enumerate(dataloader):
    if i == 0:
        print(f"首批: x {tuple(x_batch.shape)}, y {tuple(y_batch.shape)}")
        print(f"标签分布: {y_batch.tolist()}")
        break

## 6. GPU 加速

把张量和模型搬到 GPU，运算大幅加速。关键是 `.to(device)` 统一设备。

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"可用设备: {device}")

if device.type == "cuda":
    # 搬到 GPU
    x = torch.randn(1000, 1000).to(device)
    model = nn.Linear(1000, 1000).to(device)
    # 运算在 GPU 上进行
    out = model(x)
    print(f"GPU 上运算，输出 device: {out.device}")
else:
    print("（无 GPU，CPU 上运行。代码逻辑一致，只是慢些。）")
    x = torch.randn(1000, 1000)
    print(f"CPU 上张量 device: {x.device}")

print("\n💡 模型和数据必须在同一设备，否则报错。.to(device) 统一管理。")

---
> **小结**：张量（数据）+ autograd（梯度）+ nn.Module（模型）+ 训练循环（优化）+ DataLoader（批次）+ GPU（加速）。
> 这套范式贯穿全书 ch02-ch07。后续章节都是在这个基础上造 GPT。